#SE NAO TIVER ENV RODA ESSE

In [1]:

# %conda create --name IBD --file environment.yml

# Importando as bibliotecas e também lendo o arquivo excel:

In [2]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import sqlite3
import re
import os
import datetime
from verify_form import *
DB_FILE = 'gas_data.db' # Para ler o banco de dados SQLite
XLSX_FILE = 'gn_marco_2025.xlsx' # Para ler o arquivo XLSX


print(f"Iniciando a extração do arquivo '{XLSX_FILE}'...")
try:
    df_raw = pd.read_excel(XLSX_FILE)
except FileNotFoundError:
    print(f"ERRO: O arquivo '{XLSX_FILE}' não foi encontrado.")
    print("Por favor, certifique-se de que o arquivo está no diretório correto.")
    raise

print("Extração concluída.")

Iniciando a extração do arquivo 'gn_marco_2025.xlsx'...
Extração concluída.


# Normalização:

In [3]:
print("Iniciando a transformação dos dados...")

# Mapeamento de colunas para nomes mais curtos e compatíveis com SQL
column_mapping = {
    'Código da Instalação de Transporte': 'codigo_instalacao_transporte',
    'Nome da Instalação de Transporte': 'nome_instalacao_transporte',
    'Nome da Instalação de Gasoduto': 'nome_instalacao_gasoduto',
    'Código da Instalação de Gasoduto': 'codigo_instalacao_gasoduto',
    'Tipo da instalação de Gasoduto': 'tipo_instalacao',
    'Nome do Município da Instalação de Gasoduto': 'municipio',
    'Nome da UF da Instalação de Gasoduto': 'uf',
    'Nome do Operador da instalação de Gasoduto': 'nome_operador',
    'Código do Operador da Instalação de Gasoduto': 'codigo_operador',
    'Nome do Carregador que usa a Instalação de Gasoduto': 'nome_carregador',
    'Código do Carregador que usa a Instalação de Gasoduto': 'codigo_carregador',
    'Nome do Contrato da Instalação de Gasoduto': 'nome_contrato',
    'Nome da Variável': 'variavel_completa'
}
df = df_raw.rename(columns=column_mapping)

Iniciando a transformação dos dados...


# UNPIVOT

In [4]:
# Identificar colunas de data para a operação de "unpivot"
date_columns = [col for col in df.columns if isinstance(col, datetime.datetime)]
id_vars = list(column_mapping.values())

#UNPIVOT (transformar de formato largo para longo)
df_long = pd.melt(df, id_vars=id_vars, value_vars=date_columns,
                  var_name='data_medicao', value_name='valor')

#convert data
df_long.replace('#N/D', np.nan, inplace=True)
df_long['valor'] = pd.to_numeric(df_long['valor'], errors='coerce')
df_long.dropna(subset=['valor'], inplace=True)
df_long['data_medicao'] = pd.to_datetime(df_long['data_medicao'])

#Extract variable name and unit
var_regex = re.compile(r'^(.*?)\s*\((.*)\)$')
extracted_vars = df_long['variavel_completa'].str.extract(var_regex)

df_long['nome_variavel'] = extracted_vars[0].str.strip()
df_long['unidade_medida'] = extracted_vars[1].str.strip()
df_long['nome_variavel'].fillna(df_long['variavel_completa'], inplace=True)
df_long['unidade_medida'].fillna('N/A', inplace=True)


## Cria DB

In [5]:
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = 0;") # Habilitar restrições de chave estrangeira

# DDL - Data Definition Language (Comandos para criar as tabelas)
ddl_scripts = """
CREATE TABLE Operador (
    codigo_operador INTEGER PRIMARY KEY,
    nome_operador TEXT NOT NULL UNIQUE
    
);

CREATE TABLE Carregador (
    codigo_carregador INTEGER PRIMARY KEY,
    nome_carregador TEXT NOT NULL UNIQUE
);



CREATE TABLE Instalacao_Transporte (
    codigo_instalacao_transporte INTEGER PRIMARY KEY,
    nome_instalacao_transporte TEXT NOT NULL UNIQUE
);

CREATE TABLE Instalacao_Gasoduto (
    codigo_instalacao_gasoduto INTEGER PRIMARY KEY,
    nome_instalacao_gasoduto TEXT NOT NULL,
    tipo_instalacao TEXT NOT NULL,
    municipio TEXT NOT NULL,
    uf TEXT NOT NULL,
    codigo_instalacao_transporte INTEGER NOT NULL,
    codigo_operador INTEGER NOT NULL,
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte),
    FOREIGN KEY (codigo_operador) REFERENCES Operador(codigo_operador)
);


CREATE TABLE Contrato (
    codigo_instalacao_transporte INTEGER NOT NULL,
    codigo_carregador INTEGER NOT NULL,
    nome_contrato TEXT,
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte),
    FOREIGN KEY (codigo_carregador) REFERENCES Carregador(codigo_carregador),
    PRIMARY KEY (codigo_instalacao_transporte, codigo_carregador,nome_contrato)
);

CREATE TABLE Medicao (
    codigo_medicao INTEGER PRIMARY KEY,
    data_medicao DATE NOT NULL,
    nome_variavel TEXT NOT NULL, 
    valor REAL NOT NULL,                                                                                                

    codigo_instalacao_gasoduto INTEGER NULL,
    FOREIGN KEY (codigo_instalacao_gasoduto) REFERENCES Instalacao_Gasoduto(codigo_instalacao_gasoduto)
);
"""

cursor.executescript(ddl_scripts)

## Popula operadores:

In [6]:
# Operador
df_operador = df_long[['codigo_operador', 'nome_operador']].drop_duplicates().dropna(subset=['codigo_operador'])
df_operador.to_sql('Operador', conn, if_exists='append', index=False)
df_operador

,codigo_operador,nome_operador
0,1717813,Gasocidente do Mato Grosso Ltda - GOM
34,3004992714,Nova Transportadora do Sudeste S.A. - NTS
7338,3006248349,Transportadora Associada de Gás S.A. - TAG
10148,5,Transportadora Brasileira Gasoduto Bolívia-Bra...
11105,3003146349,Transportadora Sulbrasileira de Gás S.A. - TSB


In [7]:
verify_database_normalization(DB_FILE,'Operador')

The table 'Operador' is in 1NF.
INFO: A tabela 'Operador' possui uma chave primária simples. Está em 2NF.
INFO: A tabela 'Operador' está em 2NF e não foram encontrados indicadores de dependência transitiva. A tabela está em 3NF.


{'Operador': {'1NF': True, '2NF': True, '3NF': True}}

## Criando o dataset de carregadores:

In [8]:
df_carregador = df_long[['codigo_carregador', 'nome_carregador']] \
    .dropna(subset=['codigo_carregador', 'nome_carregador']) \
    .drop_duplicates(subset=['codigo_carregador'])

df_carregador.to_sql('Carregador', conn, if_exists='append', index=False)
df_carregador.head()

,codigo_carregador,nome_carregador
0,1.645009e+06,AMBAR
17,6.023921e+06,MTGAS
34,3.300017e+07,Petróleo Brasileiro S.A. - PETROBRAS
1120,2.033043e+09,Companhia Siderúrgica Nacional
2148,2.232025e+06,BRAVA


In [9]:
verify_database_normalization(DB_FILE,'Carregador')

The table 'Carregador' is in 1NF.
INFO: A tabela 'Carregador' possui uma chave primária simples. Está em 2NF.
INFO: A tabela 'Carregador' está em 2NF e não foram encontrados indicadores de dependência transitiva. A tabela está em 3NF.


{'Carregador': {'1NF': True, '2NF': True, '3NF': True}}

## Criando o dataset de variaveis:

In [10]:
df_variavel = df_long[['nome_variavel', 'unidade_medida']] \
    .dropna(subset=['nome_variavel', 'unidade_medida']) \
    .drop_duplicates()

df_variavel.to_sql('Variavel', conn, if_exists='append', index=False)
df_variavel.head()

,nome_variavel,unidade_medida
0,Gás de Uso no Sistema,mil m³
1,Gás não contado,mil m³
2,Perdas Operacionais,mil m³
3,Perdas Extraordinárias,mil m³
4,Desequilíbrio Diário,mil m³


In [11]:
verify_database_normalization(DB_FILE,'Variavel')

The table does not have a primary key.


{'Variavel': {'1NF': False, '2NF': False, '3NF': False}}

## Criando o dataset da instalação de transporte:

In [12]:
df_instalacao_transporte = df_long[['codigo_instalacao_transporte', 'nome_instalacao_transporte']] \
    .dropna(subset=['codigo_instalacao_transporte', 'nome_instalacao_transporte']) \
    .drop_duplicates(subset=['nome_instalacao_transporte'])

# Normalize case and strip spaces
df_instalacao_transporte['nome_instalacao_transporte'] = df_instalacao_transporte['nome_instalacao_transporte'].str.strip().str.lower()

df_instalacao_transporte.to_sql('Instalacao_Transporte', conn, if_exists='append', index=False)
df_instalacao_transporte.head()

,codigo_instalacao_transporte,nome_instalacao_transporte
0,700521,bolívia - mato grosso lateral cuiabá
34,700543,gasduc iii
86,700525,paulínia-jacutinga
103,700540,gastau
125,514180,anel de gás


In [13]:
verify_database_normalization(DB_FILE,'Instalacao_Transporte')

The table 'Instalacao_Transporte' is in 1NF.
INFO: A tabela 'Instalacao_Transporte' possui uma chave primária simples. Está em 2NF.
INFO: A tabela 'Instalacao_Transporte' está em 2NF e não foram encontrados indicadores de dependência transitiva. A tabela está em 3NF.


{'Instalacao_Transporte': {'1NF': True, '2NF': True, '3NF': True}}

## Criando o dataset da instalação de gasoduto

In [14]:
df[['codigo_instalacao_gasoduto', 'codigo_operador']].drop_duplicates(subset=['codigo_instalacao_gasoduto', 'codigo_operador'])

,codigo_instalacao_gasoduto,codigo_operador
0,NaN,1717813
7,111567.0,1717813
12,111565.0,1717813
29,111566.0,1717813
34,NaN,3004992714
...,...,...
11105,NaN,3003146349
11112,212585.0,3003146349
11117,212586.0,3003146349
11129,212587.0,3003146349


In [15]:
df_instalacao_gasoduto = df_long[['codigo_instalacao_gasoduto', 'nome_instalacao_gasoduto', 'tipo_instalacao', 'municipio', 'uf', 'codigo_instalacao_transporte','codigo_operador']] \
    .dropna(subset=['codigo_instalacao_gasoduto']) \
    .drop_duplicates()
# Filtra para inserir apenas os gasodutos que ainda não existem

df_instalacao_gasoduto.to_sql('Instalacao_Gasoduto', conn, if_exists='append', index=False)

df_instalacao_gasoduto.head()

,codigo_instalacao_gasoduto,nome_instalacao_gasoduto,tipo_instalacao,municipio,uf,codigo_instalacao_transporte,codigo_operador
7,111567.0,Cáceres,Ponto de Recebimento,Cáceres,MT,700521,1717813
12,111565.0,Termocuiabá,Ponto de Entrega,Cuiabá,MT,700521,1717813
29,111566.0,MTGAS,Ponto de Entrega,Cuiabá,MT,700521,1717813
41,222297.0,Interconexão Campos Elíseos I (EDG Campos Elís...,Ponto de Recebimento,Duque de Caxias,RJ,700543,3004992714
46,222296.0,Interconexão TECAB (TECAB >> GASDUC III),Ponto de Recebimento,Macaé,RJ,700543,3004992714


In [16]:
verify_database_normalization(DB_FILE,'Instalacao_Transporte')

The table 'Instalacao_Transporte' is in 1NF.
INFO: A tabela 'Instalacao_Transporte' possui uma chave primária simples. Está em 2NF.
INFO: A tabela 'Instalacao_Transporte' está em 2NF e não foram encontrados indicadores de dependência transitiva. A tabela está em 3NF.


{'Instalacao_Transporte': {'1NF': True, '2NF': True, '3NF': True}}

## Contrato

In [17]:
df_contrato = df_long[['codigo_instalacao_transporte', 'codigo_carregador', 'nome_contrato']] \
    .dropna(subset=['codigo_instalacao_transporte', 'codigo_carregador','nome_contrato']) \
    .drop_duplicates(subset=['codigo_instalacao_transporte', 'codigo_carregador','nome_contrato'])
df_contrato.to_sql('Contrato', conn, if_exists='append', index=False)
df_contrato.head()

,codigo_instalacao_transporte,codigo_carregador,nome_contrato
7,700521,1645009.0,Gasocidente do Mato Grosso Ltda x AMBAR Energi...
24,700521,6023921.0,Gasocidente do Mato Grosso Ltda x Companhia Ma...
41,700543,33000167.0,GASDUC III
93,700525,33000167.0,Paulínia-Jacutinga
110,700540,33000167.0,GASTAU


In [18]:
verify_database_normalization(DB_FILE,'Contrato')

The table 'Contrato' is in 1NF.
INFO: A tabela 'Contrato' tem uma chave primária composta, mas não possui atributos não-primos. Está em 2NF.
INFO: A tabela 'Contrato' está em 2NF e não foram encontrados indicadores de dependência transitiva. A tabela está em 3NF.


{'Contrato': {'1NF': True, '2NF': True, '3NF': True}}

## Meidição

In [19]:
# Corrected code
df_medicao = df_long[['data_medicao', 'nome_variavel', 'valor', 'codigo_instalacao_gasoduto']] \
    .dropna(subset=['data_medicao', 'nome_variavel', 'valor', 'codigo_instalacao_gasoduto']) \
    .drop_duplicates(subset=['data_medicao', 'nome_variavel', 'codigo_instalacao_gasoduto']) # <-- CHANGE IS HERE

# This line should now work without error
df_medicao.to_sql('Medicao', conn, if_exists='append', index=False)
#salva o banco de dados
conn.commit()

## VERIfica BD

In [20]:
import sqlite3
import pandas as pd

# Step 1: Connect to your database
conn = sqlite3.connect(DB_FILE)  # Replace with your actual file

# Step 2: Get the list of all table names
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

# Step 3: Print rows and columns for each table
for table in tables:
    table_name = table[0]
    try:
        # Read table into pandas DataFrame
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        
        # Count rows and columns
        num_rows = df.shape[0]
        num_cols = df.shape[1]

        # Print info
        print(f"Table: {table_name}")
        print(f"  Rows: {num_rows}")
        print(f"  Columns: {num_cols}")
        print("-" * 30)
        
    except Exception as e:
        print(f"Could not read table '{table_name}': {e}")

# Step 4: Close connection
conn.close()


Table: Operador
  Rows: 5
  Columns: 2
------------------------------
Table: Carregador
  Rows: 32
  Columns: 2
------------------------------
Table: Instalacao_Transporte
  Rows: 34
  Columns: 2
------------------------------
Table: Instalacao_Gasoduto
  Rows: 201
  Columns: 7
------------------------------
Table: Contrato
  Rows: 207
  Columns: 3
------------------------------
Table: Medicao
  Rows: 31787
  Columns: 5
------------------------------
Table: Variavel
  Rows: 27
  Columns: 2
------------------------------


In [21]:
import sqlite3
import pandas as pd

db_file = 'gas_data.db'


conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Query para listar todas as tabelas no banco de dados
df = pd.read_sql_query(f"SELECT * FROM Medicao", conn)
df

,codigo_medicao,data_medicao,nome_variavel,valor,codigo_instalacao_gasoduto
0,1,2025-03-01 00:00:00,Volume Solicitado,0.00,111567
1,2,2025-03-01 00:00:00,Volume Programado,0.00,111567
2,3,2025-03-01 00:00:00,Volume Realizado,0.00,111567
3,4,2025-03-01 00:00:00,Alocação,0.00,111567
4,5,2025-03-01 00:00:00,Pressão Média,82.30,111567
...,...,...,...,...,...
31782,31783,2025-03-31 00:00:00,Volume Solicitado,800.00,212588
31783,31784,2025-03-31 00:00:00,Volume Programado,800.00,212588
31784,31785,2025-03-31 00:00:00,Volume Realizado,459.28,212588
31785,31786,2025-03-31 00:00:00,Alocação,100.00,212588
